In [6]:
import sys
from time import perf_counter

class UP:
    """
        UP includes a pattern name composed of two or more items

        Args:
            name (str): item's name.
            expSup (float): expected support of item.
            max (float): maximum probability of item.
            TID_prob_list (dict): List of transactions of the item, 
                                  where the key (str) is the TID.
                                  and the value (float) is the probability of the item in the corresponding transaction.
        """
    def __init__(self, name):
        self.name = name
        self.expSup = 0  
        self.max = 0
        self.transaction = {}  
            
    def update_UP(self, TID, prob):
        """
        Update expSup, max and transaction of an UP
        
        Args:
            TID (int): id of a transaction
            prob (float): probability of item in tid transaction
        """
        if(prob > 0):
            self.expSup += prob
            self.transaction[TID] = prob
            
            if prob > self.max:
                self.max = prob
                
    def __repr__(self):
        return f"UP_List(item={self.name}, expSup={round(self.expSup, 2)}, max={round(self.max, 2)}, transaction={self.transaction})"


class IMCUP:
    """
        The IMCUP-List is similar to a CUP-List except that it contains 2 additional parameters called Index1 and Index2.

        Args:
            name (str): item's name.
            expSup (float): expected support of item.
            max (float): maximum probability of item.
            TID_prob_list (dict): List of transactions of the item, 
                                  where the key (str) is the TID 
                                  and the value (float) is the probability of the item in the corresponding transaction.
            index1 (int): the index of the last processed tuple of the first list.
            index2 (int): the index of the last processed tuple of the second list.
        """
    def __init__(self, name):
        self.name = name
        self.expSup = 0  
        self.max = 0
        self.transaction = {}  
        self.index1 = -1
        self.index2 = -1
    
            
    def combine(self, minSup, cupOfPref, cupX, cupY): 
        """
        Combine 2 UP or IMCUP
        
        Args:
            minSup: current minSup in Tk
            cupOfPref: an UP or IMCUP whose item is the common item(s) of cupX and cupY 
            cupX, cupY: an UP or IMCUP
        """  
        global UP_List, IMCUP_List
        
        tid_x = list(cupX.transaction.keys())   # TID list in cupX transaction
        tid_y = list(cupY.transaction.keys())   # tid list in cupY transaction`
        
        while self.index1 < (len(tid_x) - 1) and self.index2 < (len(tid_y) - 1):              
            if tid_x[self.index1 + 1] < tid_y[self.index2 + 1]:
                self.index1 += 1
            elif tid_x[self.index1 + 1] > tid_y[self.index2 + 1]:
                self.index2 += 1
            else:
                # if cupX and cupY have the same TID
                self.index1 += 1
                self.index2 += 1
                
                # If cupX.name and cupY.name have no common items
                # then prob = cupX.prob * cupY.prob (with same tid in transaction)
                # If cupX and cupY have at least 1 common item
                # then prob = (cupX.prob * cupY.prob)/cupOfPref.prob (with same tid in transaction)
                prob = cupX.transaction[tid_x[self.index1]] * cupY.transaction[tid_y[self.index2]] \
                    if cupOfPref == None \
                    else (cupX.transaction[tid_x[self.index1]] * cupY.transaction[tid_y[self.index2]])/cupOfPref.transaction[tid_x[self.index1]]
                
                self.expSup += prob
                self.transaction[tid_x[self.index1]] = prob
                
                if prob > self.max:
                    self.max = prob
                
                if (self.check_ppf(minSup, cupX)):
                    return self        
        
        return self   
    
    def check_ppf(self, minSup, cupX):
        """
        Pre-pruning factor
        Formula: (minSup - cur_expSup) > (size - k) * max
        
        Operands:
            minSup: current minSup in Tk
            cur_expSup: expSup value of IMCUP (self) accumulated so far
            size: number of tuples in cupX's transaction
            k: number of tuples processed so far in cupX's transaction (0 ≤ k ≤ size)
            max:  maximum existential probability of the values of the cupX's tuples.

        Args:
            minSup: current minSup in Tk
            cupX: an UP or IMCUP        
        """
        
        size = len(cupX.transaction)   
        return minSup - self.expSup > (size - (self.index1 + 1)) * cupX.max
               
    def __repr__(self):
        return f"IMCUP_List(name={self.name}, expSup={round(self.expSup, 2)}, max={round(self.max, 2)}, transaction={self.transaction})"


class AlgorithmITUFP:
    def __init__(self):
        """
        Algorithm ITUFP.
        """
    
    def mining_TopK(self, k):
        """
        Process Top-K mining requests.

        Args:
            k (int): number of Top-K uncertain frequent patterns (UFPs) to be exploited.
        """
        global UP_List, Tk
        
        minSup = 0
        Tk = list(UP_List.values())[:k]
        U = list(UP_List.values())
        
        if(len(Tk) == k):
            minSup = Tk[-1].expSup
        
        self.Mine_Patterns(U, k, Tk, minSup)
    

    def Mine_Patterns(self, U, K, Tk, minSup):
        """
        Exploiting UFPs, combine 2 UPs.

        Args:
            U (list): list of UP.
            K (int): number of Top-K UFPs to be exploited.
            Tk (dict): dict of UFPs, key is item.name and value is UP
            minSup (float): minimum expSup in Tk.
            
        PUFP condition:
            Formula: 
                overestimated expSup of X' = expSup(X) * Max(i')
            
            Operands:
                X: an UFP --> X.name  {i1, i2, …, ik}
                X': a supper pattern of X --> X'.name = {i1, i2, …, ik, i'}
                overestimated expSup of X': maximum expSup value that X' can have
                Max(i'): maximum value among the existential probabilities that i' can have in the given uncertain database.
        """
        global IMCUP_List
        
        for ux in U:                    
            if ux.expSup > minSup:      
                item = IMCUP_List.get(ux.name, None)
                if item == None:
                    IMC = []  
                    pref = ux.name

                for uj in U[U.index(ux) + 1:]: 
                    if uj.expSup > minSup:      
                        if (ux.expSup * uj.max) > minSup:   # PUFP condition
                            c = self.isExistIMCUP(pref, ux, uj)
                            if (isinstance(c, str)):
                                c = IMCUP(c)
                                IMC.append(c)
                                IMCUP_List[c.name] = c
                                                       
                            c.combine(minSup, self.cupOfPref(pref, ux), ux, uj)
                        
                            if c.expSup > minSup:  
                                if len(Tk) < k:
                                    Tk.append(c) 
                                    
                                if len(Tk) == k:
                                    Tk[-1] = c
                                    Tk.sort(key=lambda x: x.expSup, reverse=True)   
                                    minSup = Tk[-1].expSup                                                      
                    
                if len(IMC) > 1:        
                    self.ITUFP_growth(IMC, pref, K, Tk, minSup)  
                        
    
    def ITUFP_growth(self, IMC, pref, K, Tk, minSup):
        """
        Exploiting UFPs, combine 2 IMCUPs.

        Args:
            IMC (list): list of IMCUP.
            pref (str): The prefix itemset for the current step.
            K (int): number of Top-K UFPs to be exploited.
            Tk (dict): dict of UFPs, key is item.name and value is UP
            minSup (float): minimum expSup in Tk.
            
        """
        global IMCUP_List
        IMC_1 = []
        pref_1 = ""
        
        for cx in IMC:                    
            if cx.expSup > minSup:      
                item = next((item for item in IMC_1 if item.name == cx.name), None)
                if item == None:
                    IMC_1 = []  
                    pref_1 = cx.name

                for cj in IMC[IMC.index(cx) + 1:]:  
                    if cj.expSup > minSup:      
                        if (cx.expSup * cj.max) > minSup:   # PUFP condition
                            c = self.isExistIMCUP(pref, cx, cj)
                            if (isinstance(c, str)):
                                c = IMCUP(c)
                                IMC_1.append(c)
                                IMCUP_List[c.name] = c
                            
                            c.combine(minSup, self.cupOfPref(pref, cx), cx, cj)
                        
                            if c.expSup > minSup:  
                                if(len(Tk) == k):
                                    Tk[-1] = c
                                    Tk.sort(key=lambda x: x.expSup, reverse=True)  
                                    minSup = Tk[-1].expSup
                                else:
                                    Tk.append(c)
                                                      
                if len(IMC_1) > 1:    
                    self.ITUFP_growth(IMC_1, pref_1, K, Tk, minSup) 


    def cupOfPref(self, pref, cupX):
        """
        Find UP/IMCUP for pref.

        Args:
            pref: prefix of cupX
            cupX: an UP/IMCUP
        
        Returns:    
            UP/IMCUP if UP/IMCUP is found with the same name as pref
            otherwise return None
        """
        items_in_pref = len(pref.split(","))
        items_in_cupX = len(cupX.name.split(","))
        
        if items_in_pref == 1 and items_in_cupX == 1:
            return None
        return UP_List.get(pref, None) \
            if items_in_pref == 1 \
            else IMCUP_List.get(pref, None)
                        
                        
    def isExistIMCUP(self, pref, cupX, cupY):
        """
        Check if IMCUP of cupX and cupY exists

        Args:
            pref: prefix of cupX
            cupX, cupY: an UP/IMCUP
        
        Returns:    
            IMCUP if cupX and cupY have been combined into IMCUP
            otherwise method return a new name to create IMCUP for these 2 cups
        """
        global IMCUP_List
        new_name_cupY = "".join(cupY.name.replace(pref, "", 1).split(", "))
        name_IMCUP = cupX.name + ", " + new_name_cupY
        
        item = IMCUP_List.get(name_IMCUP, None)
        return item if item != None else name_IMCUP


def readFile(path):      
    """
    Scan database

    Args:
        path(str): file path
    """
    try:
        with open(path, "r", encoding="utf-8") as file:
            data = file.read().split("\n")
            items = data[0].split(" ")
            prob = data[1:]
            
            create_UP_List(items, prob)
            
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        sys.exit(0)
    except Exception as e:
        print(f"An error occurred: {e}")
        sys.exit(0)

    
def create_UP_List(items, prob):
    """
    Create an UP

    Args:
        items (list): list of item names
        prob (list): list of probability in each transaction
    """
    global UP_List
    
    # create UP
    UP_List = {item: UP(item) for item in items}
   
    # Update UP properties
    for TID, transaction in enumerate(prob, start=1):
        for value, p in zip(UP_List.values(), map(float, transaction.split())):          
            value.update_UP(TID, p)
            
    # Sort the UP_List in descending order of expSup
    UP_List = dict(sorted(UP_List.items(), key=lambda x: x[1].expSup, reverse=True))


def main_flow(path, output_file, k):
    readFile(path)

    tufp = AlgorithmITUFP()
    tufp.mining_TopK(k)

    with open("output/" + output_file + ".txt", "w", encoding="utf-8") as file:
        for t in Tk:
            file.write(str(t) + "\n")

    print("Write file successfully")
    

UP_List = {}
IMCUP_List = {}
Tk = []

In [7]:
path = "data/example.txt"
output_file = "example"
k = 50

main_flow(path, output_file, k)

Write file successfully


In [8]:
path = "data/foodmart.txt"
output_file = "foodmart"
k = 500

main_flow(path, output_file, k)

Write file successfully
